# Explore here

Dogs vs. Cats image classification. Step 1 prepares the 25,000 training photos with the Keras image processing API: reshape each file to a 200×200 square, read the class from the filename (`cat.*` → `0.0`, `dog.*` → `1.0`), and save the `(photos, labels)` tuple.

Training images in `data/raw/train` are loaded only by Keras for this resize/label step. They are not plotted or inspected here.

In [ ]:
from app import load_saved_dataset, load_training_dataset, save_dataset

photos, labels = load_training_dataset()
print(photos.shape, labels.shape)
print(photos.dtype, labels.dtype)
print("cats (0.0):", int((labels == 0.0).sum()))
print("dogs (1.0):", int((labels == 1.0).sum()))

In [ ]:
photos_path, labels_path = save_dataset(photos, labels)
print(photos_path)
print(labels_path)

saved_photos, saved_labels = load_saved_dataset()
print(saved_photos.shape, saved_labels.shape)

## Step 2. ImageDataGenerator for train and test

`flow_from_directory` needs one subdirectory per class. Filenames from `data/raw/train` are linked into `data/interim/dataset_dogs_vs_cats/{train,test}/{cats,dogs}` (25% held out as test). Then `trdata` receives the train folder and `tsdata` receives the test folder. Photos are not plotted.

In [ ]:
from app import create_data_generators, organize_dataset

dataset_home = organize_dataset()
print(dataset_home)

trdata, traindata, tsdata, testdata = create_data_generators()
print(type(trdata).__name__, type(tsdata).__name__)
print("train:", traindata.directory, traindata.samples, traindata.class_indices)
print("test:", testdata.directory, testdata.samples, testdata.class_indices)

## Step 3. Build an ANN

EfficientNet-B0 is the convolutional backbone (trained from scratch, `weights=None`). `GlobalAveragePooling2D` flattens its feature maps, then two `Dense` layers classify cat vs dog. The generators feed **224×224** images and one-hot labels so they match `input_shape=(224, 224, 3)` and `Dense(2, softmax)`. EfficientNet already rescales 0–255 inputs, so `trdata`/`tsdata` do not divide by 255 again.

In [ ]:
from app import (
    BATCH_SIZE,
    MODEL_IMAGE_SIZE,
    build_model,
    compile_model,
    create_data_generators,
    evaluate_model,
    organize_dataset,
    save_model,
    train_model,
)

organize_dataset()
_, traindata, _, testdata = create_data_generators(
    image_size=MODEL_IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    rescale=None,
)

model = compile_model(build_model())
model.summary()

In [ ]:
history = train_model(model, traindata, testdata)
loss, accuracy = evaluate_model(model, testdata)
print(f"test loss: {loss:.4f}")
print(f"test accuracy: {accuracy:.4f}")
print(f"best val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(save_model(model))